In [5]:
import numpy as np
import pandas as pd
from brainiak.isc import isc
from voxelwise_tutorials.delayer import Delayer
from himalaya.scoring import correlation_score
from nilearn.input_data import NiftiLabelsMasker
from nilearn.datasets import fetch_atlas_schaefer_2018
from himalaya.kernel_ridge import KernelRidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import os

# ============================================================
# 設定
# ============================================================
task_train = "piemanpni"   # 学習に使う物語
task_test  = "bronx"       # 評価に使う別物語
base_dir = os.getcwd()
label_path = os.path.join(os.path.dirname(base_dir),
                          'atlas_label/Schaefer2018_400Parcels_7Networks_order_info.txt')

# ============================================================
# 7ネットワークラベル読み込み
# ============================================================
def load_network_labels_7networks(label_fn):
    networks = []
    with open(label_fn, 'r') as f:
        for line in f:
            if '7Networks_' in line:
                network_name = line.split('_')[2]
                networks.append(network_name)
    unique_network_labels = sorted(set(networks), key=networks.index)
    network_indices = [
        int(np.median([i for i, n in enumerate(networks) if n == network]))
        for network in unique_network_labels
    ]
    return networks, unique_network_labels, network_indices

networks, unique_networks, indices = load_network_labels_7networks(label_path)

# ============================================================
# Atlas
# ============================================================
n_parcels = 400
atlas = fetch_atlas_schaefer_2018(n_rois=n_parcels, yeo_networks=17, resolution_mm=2)
masker = NiftiLabelsMasker(atlas.maps, labels=atlas.labels, standardize=True)

# ============================================================
# 学習用データ（bronx）読み込み
# ============================================================
# m_name = input("使用するモデル名を入力 (例: llama, gpt2, gte): ")
model_list = ["gpt2", "gpt_oss", "llama", "llama3", "gte", "w2v"]

df_all = pd.read_csv(f"../parcels_csv/{task_train}_clean-parcels_all_subjects.csv")
valid_subs = sorted(df_all["sub"].unique())
n_TR = df_all["TR"].max() + 1
n_parcels = len([c for c in df_all.columns if c.startswith("parcel_")])

all_ts = np.zeros((len(valid_subs), n_TR, n_parcels))
for i, subname in enumerate(valid_subs):
    sub_df = df_all[df_all["sub"] == subname].sort_values("TR")
    all_ts[i] = sub_df[[f"parcel_{j}" for j in range(n_parcels)]].values
   
    

    
    
df_bronx = pd.read_csv(f"../parcels_csv/{task_test}_clean-parcels_all_subjects.csv")
valid_subs_bronx = sorted(df_bronx["sub"].unique())
n_TR_bronx = df_bronx["TR"].max() + 1
n_parcels = len([c for c in df_bronx.columns if c.startswith("parcel_")])

all_ts_bronx = np.zeros((len(valid_subs_bronx), n_TR_bronx, n_parcels))
for i, subname in enumerate(valid_subs_bronx):
    sub_df = df_bronx[df_bronx["sub"] == subname].sort_values("TR")
    all_ts_bronx[i] = sub_df[[f"parcel_{j}" for j in range(n_parcels)]].values

for m_name in model_list:
    
    
    df_all_pred = pd.read_csv(f"../predicted_all_csv/{m_name}_{task_train}_clean_predicted.csv")
    valid_subs = sorted(df_all_pred["subject"].unique())
    n_TR = df_all_pred["TR"].max() + 1
    n_parcels = len([c for c in df_all_pred.columns if c.startswith("parcel_")])
    
    all_ts_pred = np.zeros((len(valid_subs), n_TR, n_parcels))
    for i, subname in enumerate(valid_subs):
        sub_df = df_all_pred[df_all_pred["subject"] == subname].sort_values("TR")
        all_ts_pred[i] = sub_df[[f"parcel_{j}" for j in range(n_parcels)]].values
    
    
    df_bronx_pred = pd.read_csv(f"../predicted_all_csv/{m_name}_{task_train}_to_{task_test}_clean_predicted.csv")
    valid_subs_bronx = sorted(df_bronx_pred["subject"].unique())
    n_TR_bronx = df_bronx_pred["TR"].max() + 1
    n_parcels = len([c for c in df_bronx_pred.columns if c.startswith("parcel_")])
    
    all_ts_bronx_pred = np.zeros((len(valid_subs_bronx), n_TR_bronx, n_parcels))
    for i, subname in enumerate(valid_subs_bronx):
        sub_df = df_bronx_pred[df_bronx_pred["subject"] == subname].sort_values("TR")
        all_ts_bronx_pred[i] = sub_df[[f"parcel_{j}" for j in range(n_parcels)]].values
    
    
    print("✅ all_ts shape =", all_ts.shape)
    print("✅ all_ts shape =", all_ts_pred.shape)
    # subjects, TR, parcel = all_ts.shape
    # all_ts_2d = all_ts.reshape(subjects, TR * parcel)
    
    # scaler = preprocessing.MinMaxScaler()
    # # scaler = MinMaxScaler()
    # all_ts_scaled_2d = scaler.fit_transform(all_ts_2d)
    
    # all_ts_scaled = all_ts_scaled_2d.reshape(subjects, TR, parcel)
    # subjects, TR, parcel = all_ts_bronx.shape
    # all_ts_bronx_2d = all_ts_bronx.reshape(subjects, TR * parcel)
    # # scaler = MinMaxScaler()
    # all_ts_bronx_scaled_2d = scaler.fit_transform(all_ts_bronx_2d)
    # all_ts_bronx_scaled = all_ts_bronx_scaled_2d.reshape(subjects, TR, parcel)
    # ============================================================
    # ISC 計算
    # ============================================================
    all_ts_t = np.transpose(all_ts, (1, 2, 0))
    isc_all = isc(all_ts_t, pairwise=False)
    
    # ============================================================
    # モデル設定
    # ============================================================
    outer_cv = KFold(n_splits=2)
    inner_cv = KFold(n_splits=5)
    
    # Mean-center each feature (columns of predictor matrix)
    scaler = StandardScaler(with_mean=True, with_std=True)
    
    # Create delays at 3, 4.5, 6, 7.5 seconds (1.5 s TR)
    delayer = Delayer(delays=[2, 3, 4, 5])
    
    # Ridge regression with alpha grid and nested CV
    alphas = np.logspace(1, 10, 10)
    #alphas = np.logspace(-3,4,20)
    ridge = KernelRidgeCV(alphas=alphas, cv=inner_cv)
    
    # Chain transfroms and estimator into pipeline
    pipeline = make_pipeline(scaler, delayer, ridge)
    
    
    # ============================================================
    # 被験者ループ
    # ============================================================
    
    results = []
    for idx_self, subname in enumerate(valid_subs):
        print(f"=== Subject: {subname} ({idx_self+1}/{len(valid_subs)}) ===")
    
        # ====== 学習物語 (bronx) ======
        other_idx = [i for i in range(len(valid_subs)) if i != idx_self]
        others_mean = all_ts[other_idx].mean(axis=0)
        Y_parcels = all_ts[idx_self]  # (TR, parcels)
        Y_predicted = all_ts_pred[idx_self]
        # 各被験者の埋め込み（bronx）
    
        # ---- Cross-validation training ----
        # Y_predicted = []
        # for train, test in outer_cv.split(Y_parcels):
        #     pipeline.fit(X[train], Y_parcels[train])
        #     pred = pipeline.predict(X[test])
        #     Y_predicted.append(pred)
        # Y_predicted = np.vstack(Y_predicted)
    
        corr_each_other = []
        for s in other_idx:
            Y_other_s = all_ts[s]  # (TR, parcels)
            corr_s = correlation_score(Y_other_s, Y_predicted)  # (parcels,)
            corr_each_other.append(corr_s)
    
        pred_vs_group = np.mean(corr_each_other, axis=0)
        #pred_vs_group = correlation_score(others_mean, Y_predicted)
        subj_vs_group = isc_all[idx_self]
        pred_vs_actual = correlation_score(Y_parcels, Y_predicted)
    
        # ============================================================
        # 別物語 (bronx) の評価
        # ============================================================
        
        
        if subname in valid_subs_bronx:
            idx_bronx = valid_subs_bronx.index(subname)
            # others_bronx_mean = all_ts_bronx[other_idx].mean(axis=0)
            Y_parcels_neo = all_ts_bronx[idx_self]
            Y_pred_neo = all_ts_bronx_pred[idx_self]
    
    
            corr_each_other_story = []
            
    
            for s in other_idx:
                Y_other_s_neo = all_ts_bronx[s]  # (TR, parcels)
                corr_s_neo = correlation_score(Y_other_s_neo, Y_pred_neo)
                corr_each_other_story.append(corr_s_neo)
                
            pred_vs_bronx_group = np.mean(corr_each_other_story, axis=0)
            #pred_vs_bronx_group = correlation_score(others_bronx_mean, Y_pred_neo)
            pred_vs_other_story = correlation_score(Y_parcels_neo, Y_pred_neo)
        else:
            print(f"Warning: {subname} not found in bronx dataset, filling with NaN")
            pred_vs_other_story = np.full(n_parcels, np.nan)
    
        # ============================================================
        # DataFrameにまとめる
        # ============================================================
        df_sub = pd.DataFrame({
            "subject": subname,
            "parcel": np.arange(n_parcels),
            "pred_vs_group": pred_vs_group,
            "subj_vs_group": subj_vs_group,
            "pred_vs_actual": pred_vs_actual,
            "pred_vs_other_story": pred_vs_other_story,
            "pred_vs_other_story_group": pred_vs_bronx_group,
            "network": networks
        })
        results.append(df_sub)
    
    # ============================================================
    # 出力
    # ============================================================
    df_allsubs = pd.concat(results, ignore_index=True)
    output_path = f"../analysis_corr/{m_name}_{task_train}_allsubjects_with_{task_test}_clean_corr_df.csv"
    df_allsubs.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"✅ 出力完了: {output_path}")

print("\n\n🎉 全モデルの処理が完了しました！")

[fetch_atlas_schaefer_2018] Dataset found in /home/y-sato/nilearn_data/schaefer_2018
✅ all_ts shape = (46, 267, 400)
✅ all_ts shape = (46, 267, 400)
=== Subject: sub-127 (1/46) ===
=== Subject: sub-265 (2/46) ===
=== Subject: sub-267 (3/46) ===
=== Subject: sub-272 (4/46) ===
=== Subject: sub-273 (5/46) ===
=== Subject: sub-274 (6/46) ===
=== Subject: sub-275 (7/46) ===
=== Subject: sub-276 (8/46) ===
=== Subject: sub-277 (9/46) ===
=== Subject: sub-278 (10/46) ===
=== Subject: sub-279 (11/46) ===
=== Subject: sub-281 (12/46) ===
=== Subject: sub-282 (13/46) ===
=== Subject: sub-283 (14/46) ===
=== Subject: sub-284 (15/46) ===
=== Subject: sub-285 (16/46) ===
=== Subject: sub-286 (17/46) ===
=== Subject: sub-287 (18/46) ===
=== Subject: sub-288 (19/46) ===
=== Subject: sub-289 (20/46) ===
=== Subject: sub-290 (21/46) ===
=== Subject: sub-291 (22/46) ===
=== Subject: sub-292 (23/46) ===
=== Subject: sub-293 (24/46) ===
=== Subject: sub-294 (25/46) ===
=== Subject: sub-295 (26/46) ===
==

In [7]:
Y_parcels_neo.shape

(358, 400)

In [3]:
import numpy as np
import pandas as pd
from brainiak.isc import isc
from voxelwise_tutorials.delayer import Delayer
from himalaya.scoring import correlation_score
from nilearn.input_data import NiftiLabelsMasker
from nilearn.datasets import fetch_atlas_schaefer_2018
from himalaya.kernel_ridge import KernelRidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import os

task_train = "piemanpni"   # 学習に使う物語
task_test  = "bronx"       # 評価に使う別物語
base_dir = os.getcwd()
label_path = os.path.join(os.path.dirname(base_dir),
                          'atlas_label/Schaefer2018_400Parcels_7Networks_order_info.txt')

df_all = pd.read_csv(f"../parcels_csv/{task_train}_clean-parcels_all_subjects.csv")
valid_subs = sorted(df_all["sub"].unique())
n_TR = df_all["TR"].max() + 1
n_parcels = len([c for c in df_all.columns if c.startswith("parcel_")])

all_ts = np.zeros((len(valid_subs), n_TR, n_parcels))
for i, subname in enumerate(valid_subs):
    sub_df = df_all[df_all["sub"] == subname].sort_values("TR")
    all_ts[i] = sub_df[[f"parcel_{j}" for j in range(n_parcels)]].values
   
    

    
    
df_bronx = pd.read_csv(f"../parcels_csv/{task_test}_clean-parcels_all_subjects.csv")
valid_subs_bronx = sorted(df_bronx["sub"].unique())
n_TR_bronx = df_bronx["TR"].max() + 1
n_parcels = len([c for c in df_bronx.columns if c.startswith("parcel_")])

all_ts_bronx = np.zeros((len(valid_subs_bronx), n_TR_bronx, n_parcels))
for i, subname in enumerate(valid_subs_bronx):
    sub_df = df_bronx[df_bronx["sub"] == subname].sort_values("TR")

    
all_ts_t = np.transpose(all_ts, (1, 2, 0))
isc_all = isc(all_ts_t, pairwise=False)

In [4]:
isc_all.shape

(46, 400)

In [5]:
isc_all

array([[-0.07513013,  0.11323232,  0.11072415, ..., -0.13444242,
         0.19988087,  0.2250967 ],
       [ 0.01501299,  0.10097839,  0.03397993, ...,  0.09557084,
         0.10114141,  0.03492447],
       [ 0.20541571,  0.30959195,  0.121092  , ...,  0.03805318,
         0.27088295,  0.27092691],
       ...,
       [ 0.12777172,  0.30855328,  0.03345277, ...,  0.12156085,
         0.18908309,  0.13618971],
       [ 0.21304424,  0.42401023,  0.02283586, ...,  0.23430827,
         0.22830566,  0.26180133],
       [ 0.13674477,  0.1019073 , -0.00236194, ...,  0.07802054,
         0.29290071,  0.40179051]])